# CNN Assignment — Reading the Road: Traffic Sign Recognition

**Level:** Intermediate · **Suggested time:** 1–2 weeks · **Frameworks:** TensorFlow/Keras *or* PyTorch

## The scenario
You have joined the perception team at a self-driving car startup. The car's camera captures the road, and your job is to build a CNN that reads a cropped photo of a single traffic sign and says *which of 43 signs it is*. The data (GTSRB) is genuinely messy — variable size, lighting, blur, and class imbalance — and that is exactly where the interesting decisions live.

> **How this notebook works:** each step gives you a little starter code, a set of **Hints**, and a **TODO** checklist. You write the rest. Pick one framework and stick with it. Set a seed, use a GPU (Colab: *Runtime → Change runtime type → GPU*).


### Setup
**Hints**
- Import only what your chosen framework needs; delete the other block.
- Fix every random seed you can (`numpy`, and your framework's) so results are reproducible.

**TODO**
- [ ] Import your libraries
- [ ] Set a global `SEED`
- [ ] Detect and print whether a GPU is available


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

# TODO: import your framework and set its seed; detect GPU
# PyTorch:  import torch, torchvision ; torch.manual_seed(SEED)
# TF/Keras: import tensorflow as tf ; tf.random.set_seed(SEED)
import random
import torch, torchvision
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
import torchvision.transforms.functional as TF

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| GPU available:", torch.cuda.is_available(), "| device:", DEVICE)


---
## Part A — Explore the data  *(10%)*

**Hints**
- Load with `torchvision.datasets.GTSRB(root, split='train', download=True)` **or** `tensorflow_datasets.load('gtsrb')`, or a Kaggle folder/CSV.
- To build a label list fast in torchvision, the `(path, label)` pairs live in `dataset._samples`.
- Imbalance matters later (Part D): note which classes are rare *now*.
- Don't load every image just to measure sizes — sample a few hundred.

**TODO**
- [ ] A1: load train + test, print how many images in each and how many classes
- [ ] A2: show a 5×5 grid of random training images **with their labels**
- [ ] A3: plot a histogram of class frequencies; name the 3 rarest classes
- [ ] A4: report min / median / max image width and height


In [ ]:
# A1
# TODO: load the datasets, then set NUM_CLASSES and print the counts
NUM_CLASSES = 43
from collections import Counter

raw_train = torchvision.datasets.GTSRB(root="data", split="train", download=True)
raw_test  = torchvision.datasets.GTSRB(root="data", split="test",  download=True)

train_labels = np.array([lbl for _, lbl in raw_train._samples])
test_labels  = np.array([lbl for _, lbl in raw_test._samples])
NUM_CLASSES = len(np.unique(train_labels))
print(f"Train images: {len(raw_train)} | Test images: {len(raw_test)} | Classes: {NUM_CLASSES}")

CLASS_NAMES = [
    "Speed limit 20", "Speed limit 30", "Speed limit 50", "Speed limit 60", "Speed limit 70",
    "Speed limit 80", "End speed limit 80", "Speed limit 100", "Speed limit 120", "No passing",
    "No passing (>3.5t)", "Right-of-way at junction", "Priority road", "Yield", "Stop",
    "No vehicles", "No vehicles >3.5t", "No entry", "General caution", "Dangerous curve left",
    "Dangerous curve right", "Double curve", "Bumpy road", "Slippery road", "Road narrows right",
    "Road work", "Traffic signals", "Pedestrians", "Children crossing", "Bicycles crossing",
    "Ice/snow", "Wild animals crossing", "End all limits", "Turn right ahead", "Turn left ahead",
    "Ahead only", "Go straight or right", "Go straight or left", "Keep right", "Keep left",
    "Roundabout", "End no passing", "End no passing (>3.5t)",
]


In [ ]:
# A2 — grid of 25 random images with labels
# Hint: fig, axes = plt.subplots(5, 5, ...); loop, pick a random index, imshow, set_title(label)
# TODO
rng = np.random.RandomState(SEED)
idxs = rng.choice(len(raw_train), 25, replace=False)
fig, axes = plt.subplots(5, 5, figsize=(11, 12))
for ax, i in zip(axes.ravel(), idxs):
    img, lbl = raw_train[i]
    ax.imshow(img)
    ax.set_title(f"{lbl}: {CLASS_NAMES[lbl][:20]}", fontsize=7)
    ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# A3 — class-frequency histogram
# Hint: from collections import Counter; count your label list; plt.bar(...)
# TODO
counts = Counter(train_labels.tolist())
freq = np.array([counts[c] for c in range(NUM_CLASSES)])
plt.figure(figsize=(12, 4))
plt.bar(range(NUM_CLASSES), freq)
plt.xlabel("Class id"); plt.ylabel("# training images"); plt.title("Class frequencies (train)")
plt.show()

rare_classes = np.argsort(freq)[:3].tolist()
print("3 rarest classes:")
for c in rare_classes:
    print(f"  class {c:2d} ({CLASS_NAMES[c]}): {freq[c]} images")
print(f"Most common class has {freq.max()} images -> imbalance ratio ~{freq.max()/freq.min():.1f}x")


In [ ]:
# A4 — image size distribution (sample a few hundred images)
# Hint: a PIL image exposes .size as (width, height)
# TODO
sample_idx = np.random.RandomState(SEED).choice(len(raw_train), 500, replace=False)
sizes = np.array([raw_train[i][0].size for i in sample_idx])   # (width, height)
for name, col in [("width", 0), ("height", 1)]:
    v = sizes[:, col]
    print(f"{name:6s} min={v.min()}  median={int(np.median(v))}  max={v.max()}")


---
## Part B — Preprocess  *(10%)*

**Hints**
- Resize everything to one square size (32×32 or 48×48 are sensible). **Justify your choice in the report.**
- Scale pixels into [0,1] (torchvision `ToTensor()` already does this) or standardize per channel.
- Make the val split **stratified by class** so rare signs don't vanish from validation. A plain random split can drop a rare class entirely.
- Shuffle the *training* loader only; keep val/test in fixed order.

**TODO**
- [ ] B1/B2: define a resize+normalize transform (and a separate augmenting one later, in Part D)
- [ ] B3: build a stratified 80/20 train/val split
- [ ] B4: create train / val / test loaders (or `tf.data` pipelines) with batching


In [ ]:
# B1 + B2 — transforms / preprocessing
IMG_SIZE = 32
BATCH_SIZE = 64
# TODO: build your resize + to-tensor/normalize transform, and re-load the datasets with it
# 32x32: small signs are mostly <=~60px anyway, so little detail is lost, and it keeps training fast.
MEAN, STD = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)   # fixed constants -> no test-data statistics used
base_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),                      # scales to [0,1]
    transforms.Normalize(MEAN, STD),            # -> [-1,1]
])
# augmenting transform (used in Part D only; NO horizontal flip)
aug_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2),
    transforms.RandomAffine(0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
train_eval_ds = torchvision.datasets.GTSRB(root="data", split="train", transform=base_tf)
train_aug_ds  = torchvision.datasets.GTSRB(root="data", split="train", transform=aug_tf)
test_ds       = torchvision.datasets.GTSRB(root="data", split="test",  transform=base_tf)


In [ ]:
# B3 — stratified 80/20 train/val split
# Hint: group training indices by class, shuffle each group, take 20% of EACH class for val.
#       from collections import defaultdict; then torch Subset / sklearn train_test_split(stratify=...)
# TODO
from sklearn.model_selection import train_test_split
all_idx = np.arange(len(train_labels))
train_idx, val_idx = train_test_split(all_idx, test_size=0.2, stratify=train_labels, random_state=SEED)
print(f"train: {len(train_idx)} | val: {len(val_idx)}")
print("classes missing from val:", set(range(NUM_CLASSES)) - set(train_labels[val_idx].tolist()))


In [ ]:
# B4 — data loaders
# Hint (PyTorch): DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True) for train; shuffle=False otherwise
# TODO
NUM_WORKERS = 2
g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(Subset(train_eval_ds, train_idx), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, generator=g)
val_loader   = DataLoader(Subset(train_eval_ds, val_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
# augmented version of the SAME train split (same indices; val/test stay un-augmented)
train_aug_loader = DataLoader(Subset(train_aug_ds, train_idx), batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, generator=g)
xb, yb = next(iter(train_loader))
print("batch:", xb.shape, yb.shape, "| value range:", xb.min().item(), xb.max().item())


---
## Part C — Baseline CNN  *(25%)*

**Hints**
- Keep the baseline small: 2 blocks of Conv→ReLU→MaxPool, then Flatten→Dense→output.
- Output layer has **43 units**. In PyTorch leave them as raw logits (`CrossEntropyLoss` applies softmax internally); in Keras use `softmax` + `sparse_categorical_crossentropy`.
- Work out the flattened size: with IMG_SIZE=32 and two 2×2 pools you get 8×8 feature maps.
- Record **both** train and val metrics every epoch — you need them for the curves.

**TODO**
- [ ] C1: define the baseline CNN from scratch (no pretrained weights)
- [ ] C2: write/compile a training loop that logs train + val loss & accuracy per epoch
- [ ] C3: plot the learning curves
- [ ] C4: report **test accuracy** and interpret the curves (2–3 sentences)


In [ ]:
# C1 — baseline architecture (fill in the layers)
# PyTorch sketch:
# class BaselineCNN(nn.Module):
#     def __init__(self, n_classes=NUM_CLASSES):
#         super().__init__()
#         self.features = nn.Sequential( ... )     # TODO: Conv2d/ReLU/MaxPool x2
#         self.classifier = nn.Sequential( ... )   # TODO: Flatten/Linear/ReLU/Linear
#     def forward(self, x): ...
# TODO
class BaselineCNN(nn.Module):
    # defaults = the plain baseline; bn/drop/channels are switched on only in Part D experiments
    def __init__(self, n_classes=NUM_CLASSES, channels=(32, 64), bn=False, drop=0.0, hidden=128):
        super().__init__()
        layers, in_ch = [], 3
        for out_ch in channels:
            layers.append(nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1))
            if bn:
                layers.append(nn.BatchNorm2d(out_ch))
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            in_ch = out_ch
        self.features = nn.Sequential(*layers)
        side = IMG_SIZE // (2 ** len(channels))          # 32 -> 16 -> 8 for the baseline
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_ch * side * side, hidden), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(hidden, n_classes),                # raw logits
        )
    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(SEED)
baseline = BaselineCNN().to(DEVICE)
print(baseline)
print("parameters:", sum(p.numel() for p in baseline.parameters()))


In [ ]:
# C2 — train the baseline, logging train & val metrics each epoch
EPOCHS = 15
# Hint (PyTorch): one function run_epoch(loader, train=bool) that returns (loss, acc);
#   remember model.train()/model.eval(), optimizer.zero_grad(), loss.backward(), optimizer.step()
# TODO
import copy, time

def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

def fit(model, tr_loader, va_loader, epochs=EPOCHS, lr=1e-3, class_weights=None, cosine=False, tag=""):
    """Trains, logs train+val loss/acc per epoch, restores the best-val-accuracy weights."""
    tr_crit = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE) if class_weights is not None else None)
    va_crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs) if cosine else None
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_acc, best_state = -1, None
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = run_epoch(model, tr_loader, tr_crit, opt)
        vl, va = run_epoch(model, va_loader, va_crit)
        if sched: sched.step()
        for k, v in zip(hist, (tl, vl, ta, va)): hist[k].append(v)
        if va > best_acc:
            best_acc, best_state = va, copy.deepcopy(model.state_dict())
        print(f"[{tag}] ep {ep:2d}/{epochs} | train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f} | {time.time()-t0:.0f}s")
    model.load_state_dict(best_state)
    hist["best_val_acc"] = best_acc
    return hist

baseline_hist = fit(baseline, train_loader, val_loader, tag="baseline")


In [ ]:
# C3 — plot loss and accuracy curves (train vs val)
# TODO
def plot_history(h, title=""):
    ep = range(1, len(h["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ep, h["train_loss"], label="train"); ax[0].plot(ep, h["val_loss"], label="val")
    ax[0].set_title(f"{title} loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(ep, h["train_acc"], label="train"); ax[1].plot(ep, h["val_acc"], label="val")
    ax[1].set_title(f"{title} accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()

plot_history(baseline_hist, "Baseline")


In [ ]:
# C4 — evaluate on the test set and print accuracy
# TODO
@torch.no_grad()
def predict(model, loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        ps.append(model(x.to(DEVICE)).argmax(1).cpu()); ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()

yt, yp = predict(baseline, test_loader)
baseline_test_acc = (yt == yp).mean()
print(f"Baseline TEST accuracy: {baseline_test_acc:.4f}")

# quick numeric help for the interpretation below
h = baseline_hist
gap = h["train_acc"][-1] - h["val_acc"][-1]
print(f"final train acc {h['train_acc'][-1]:.4f} | final val acc {h['val_acc'][-1]:.4f} | gap {gap:.4f}")
print(f"val loss: min {min(h['val_loss']):.4f} at epoch {int(np.argmin(h['val_loss']))+1}, last {h['val_loss'][-1]:.4f}")


**C4 interpretation (write here):** _Overfitting, underfitting, or healthy? What in the curves tells you?_

---
## Part D — Make it better  *(30%)*

Apply **at least three** techniques and measure **each** against the baseline.

**Hints**
- Change **one thing at a time** or you won't know what helped.
- Augmentation ideas: rotation, small shifts, zoom, brightness. **Do NOT flip horizontally** — it can turn a left-turn sign into a right-turn sign.
- Overfitting (val ≪ train)? → augmentation, dropout, batchnorm. Underfitting? → bigger/deeper net, train longer.
- Imbalance: pass class weights to the loss, or oversample rare classes.
- Transfer learning (MobileNetV2 / ResNet) is a strong comparison point.

**TODO**
- [ ] Pick ≥3 techniques and run them, logging val accuracy for each
- [ ] Fill in the comparison table below (**this table is the heart of the assignment**)
- [ ] Keep your best model for Part E


In [ ]:
# D — experiments (one change at a time)
# Hint (PyTorch augmentation, no flip):
#   transforms.Compose([Resize, RandomRotation(15), ColorJitter(0.2,0.2),
#                        RandomAffine(0, translate=(0.1,0.1)), ToTensor()])
# TODO
results, models = {}, {}
results["Baseline (Part C)"] = baseline_hist["best_val_acc"]; models["Baseline (Part C)"] = baseline

# class weights for the imbalance experiment (computed from TRAIN split only)
tr_counts = np.bincount(train_labels[train_idx], minlength=NUM_CLASSES)
cw = torch.tensor(len(train_idx) / (NUM_CLASSES * tr_counts), dtype=torch.float32)

# Exp 1 - data augmentation only (same architecture as baseline)
torch.manual_seed(SEED); m1 = BaselineCNN().to(DEVICE)
h1 = fit(m1, train_aug_loader, val_loader, tag="Exp1 augment")
results["Exp 1: augmentation (rot/shift/zoom/jitter, no flip)"] = h1["best_val_acc"]; models["Exp 1: augmentation (rot/shift/zoom/jitter, no flip)"] = m1

# Exp 2 - BatchNorm + Dropout only (plain data)
torch.manual_seed(SEED); m2 = BaselineCNN(bn=True, drop=0.3).to(DEVICE)
h2 = fit(m2, train_loader, val_loader, tag="Exp2 BN+drop")
results["Exp 2: BatchNorm + Dropout(0.3)"] = h2["best_val_acc"]; models["Exp 2: BatchNorm + Dropout(0.3)"] = m2

# Exp 3 - class-weighted loss only
torch.manual_seed(SEED); m3 = BaselineCNN().to(DEVICE)
h3 = fit(m3, train_loader, val_loader, class_weights=cw, tag="Exp3 weights")
results["Exp 3: class-weighted loss"] = h3["best_val_acc"]; models["Exp 3: class-weighted loss"] = m3

# Final - combine what helps: deeper net + BN + Dropout + augmentation, longer, cosine LR
FINAL_EPOCHS = 25
torch.manual_seed(SEED); mf = BaselineCNN(channels=(32, 64, 128), bn=True, drop=0.4, hidden=256).to(DEVICE)
hf = fit(mf, train_aug_loader, val_loader, epochs=FINAL_EPOCHS, cosine=True, tag="Final")
results["Final model"] = hf["best_val_acc"]; models["Final model"] = mf
plot_history(hf, "Final model")

# results table (copy these numbers into the markdown table below)
base = results["Baseline (Part C)"]
print("\n| Experiment | Change made | Val accuracy | Δ vs baseline |\n|---|---|---|---|")
print(f"| Baseline (Part C) | — | {base:.4f} | — |")
for k, v in results.items():
    if k.startswith("Baseline"): continue
    print(f"| {k.split(':')[0]} | {k.split(':')[-1].strip() if ':' in k else 'Deeper (3 blocks) + BN + Dropout 0.4 + augmentation + cosine LR, 25 epochs'} | {v:.4f} | {v-base:+.4f} |")

# keep the best model (chosen on VALIDATION accuracy, never on test)
best_name = max(results, key=results.get)
best_model = models[best_name]
torch.save(best_model.state_dict(), "best_model.pt")
print("\nBest model:", best_name, f"(val acc {results[best_name]:.4f})")


### Part D results table (fill in)

| Experiment | Change made | Val accuracy | Δ vs baseline |
|---|---|---|---|
| Baseline (Part C) | — | ___ | — |
| Exp 1 | ___ | ___ | ___ |
| Exp 2 | ___ | ___ | ___ |
| Exp 3 | ___ | ___ | ___ |
| **Final model** | ___ | ___ | ___ |


---
## Part E — Evaluate & analyze errors  *(15%)*

**Hints**
- First collect `(y_true, y_pred)` over the whole test set, then everything else is easy.
- `sklearn.metrics.confusion_matrix` and `classification_report` do the heavy lifting.
- "Most-confused pairs" = the largest **off-diagonal** entries of the confusion matrix.
- Look at the actual misclassified images — are the confused signs genuinely similar, or just rare/low-res?

**TODO**
- [ ] E1: compute and display the confusion matrix on the test set
- [ ] E2: find the 3 most-confused class pairs and show example misclassified images
- [ ] E3: explain *why* those confusions happen
- [ ] E4: report per-class accuracy; did the rare classes from A3 suffer?


In [ ]:
# E — gather predictions, then confusion matrix + per-class report
# Hint: loop the test loader collecting argmax predictions and true labels into two arrays.
#       from sklearn.metrics import confusion_matrix, classification_report
# TODO
from sklearn.metrics import confusion_matrix, classification_report

y_true, y_pred = predict(best_model, test_loader)
final_test_acc = (y_true == y_pred).mean()
print(f"FINAL TEST ACCURACY ({best_name}): {final_test_acc:.4f}")

# E1 - confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
plt.figure(figsize=(10, 9))
plt.imshow(np.log1p(cm), cmap="viridis"); plt.colorbar(label="log(1+count)")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix (test)")
plt.show()

# E2 - 3 most-confused pairs (largest off-diagonal entries)
off = cm.copy(); np.fill_diagonal(off, 0)
top = np.dstack(np.unravel_index(np.argsort(off.ravel())[::-1][:3], off.shape))[0]
test_imgs_idx = np.arange(len(y_true))
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for r, (t, p) in enumerate(top):
    print(f"true {t} ({CLASS_NAMES[t]}) -> predicted {p} ({CLASS_NAMES[p]}): {off[t, p]} times")
    wrong = test_imgs_idx[(y_true == t) & (y_pred == p)][:4]
    for c in range(4):
        ax = axes[r, c]; ax.axis("off")
        if c < len(wrong):
            ax.imshow((test_ds[wrong[c]][0] * 0.5 + 0.5).permute(1, 2, 0).clamp(0, 1).numpy())
            ax.set_title(f"true {t} / pred {p}", fontsize=8)
plt.tight_layout(); plt.show()

# E4 - per-class accuracy (+ full report)
per_class_acc = cm.diagonal() / cm.sum(1).clip(min=1)
print("\nWorst 10 classes by test accuracy:")
for c in np.argsort(per_class_acc)[:10]:
    print(f"  class {c:2d} {CLASS_NAMES[c]:28s} acc {per_class_acc[c]:.3f}  (train imgs: {freq[c]})")
print("\nRare classes from A3:")
for c in rare_classes:
    print(f"  class {c:2d} {CLASS_NAMES[c]:28s} acc {per_class_acc[c]:.3f}  (train imgs: {freq[c]})")
print(f"mean per-class acc, rare-3: {per_class_acc[rare_classes].mean():.3f} | all classes: {per_class_acc.mean():.3f}")
print("\n", classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3, zero_division=0))


**E3 + E4 analysis (write here):** _Why are those pairs confused? Did rare classes suffer?_

---
## Part F — Stretch goals  *(bonus +10%)*

**Hints**
- Robustness: wrap the test images with added Gaussian noise / blur / brightness shift and re-measure accuracy; plot accuracy vs. corruption strength.
- Explainability: Grad-CAM highlights the pixels that drove the prediction.
- Demo: softmax the logits for one image and print the top-3 classes with confidences.

**TODO (pick one or more)**
- [ ] Robustness curve  ·  [ ] Grad-CAM heatmap  ·  [ ] Top-3 single-image demo


In [ ]:
# F — your chosen stretch goal(s)
# TODO
# collect the test set as one tensor for corruption / demo experiments
X_test = torch.cat([x for x, _ in test_loader]); y_test = torch.tensor(test_labels)

@torch.no_grad()
def predict_tensor(model, X, bs=512):
    model.eval()
    return torch.cat([model(X[i:i+bs].to(DEVICE)).argmax(1).cpu() for i in range(0, len(X), bs)])

def to01(x):  return (x * 0.5 + 0.5).clamp(0, 1)
def norm(x):  return (x - 0.5) / 0.5

# ---- F1: robustness curves (accuracy vs corruption strength) ----
torch.manual_seed(SEED)
corruptions = {
    "Gaussian noise (sigma)": ([0, 0.05, 0.1, 0.2, 0.3], lambda x, s: x + torch.randn_like(x) * s),
    "Gaussian blur (sigma)":  ([0.1, 0.5, 1.0, 1.5, 2.0], lambda x, s: TF.gaussian_blur(x, kernel_size=7, sigma=s)),
    "Brightness factor":      ([0.3, 0.5, 0.75, 1.0, 1.5], lambda x, s: x * s),
}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, (levels, fn)) in zip(axes, corruptions.items()):
    accs = []
    for s in levels:
        Xc = norm(fn(to01(X_test), s).clamp(0, 1))
        accs.append((predict_tensor(best_model, Xc) == y_test).float().mean().item())
    ax.plot(levels, accs, marker="o"); ax.set_title(name); ax.set_ylabel("test accuracy"); ax.set_ylim(0, 1.02)
plt.tight_layout(); plt.show()

# ---- F2: top-3 single-image demo ----
i = 123
best_model.eval()
with torch.no_grad():
    probs = torch.softmax(best_model(X_test[i:i+1].to(DEVICE)), 1)[0].cpu()
top3 = probs.topk(3)
plt.imshow(to01(X_test[i]).permute(1, 2, 0).numpy()); plt.axis("off"); plt.title(f"true: {CLASS_NAMES[y_test[i]]}"); plt.show()
for p, c in zip(top3.values, top3.indices):
    print(f"  {CLASS_NAMES[c]:28s} {p.item()*100:5.1f}%")

# ---- F3: Grad-CAM on the last conv layer ----
def grad_cam(model, x):
    model.eval()
    last_conv = [m for m in model.features if isinstance(m, nn.Conv2d)][-1]
    store = {}
    def fwd_hook(m, i, o):
        store["a"] = o
        o.register_hook(lambda grad: store.__setitem__("g", grad))
    h = last_conv.register_forward_hook(fwd_hook)
    logits = model(x.unsqueeze(0).to(DEVICE))
    c = logits.argmax(1).item()
    model.zero_grad(); logits[0, c].backward(); h.remove()
    w = store["g"].mean((2, 3), keepdim=True)
    cam = torch.relu((w * store["a"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[0, 0].detach().cpu()
    return (cam / (cam.max() + 1e-8)).numpy(), c

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for k, idx in enumerate([5, 400, 2000, 8000]):
    cam, c = grad_cam(best_model, X_test[idx])
    axes[0, k].imshow(to01(X_test[idx]).permute(1, 2, 0).numpy()); axes[0, k].set_title(f"pred: {CLASS_NAMES[c][:18]}", fontsize=8)
    axes[1, k].imshow(to01(X_test[idx]).permute(1, 2, 0).numpy()); axes[1, k].imshow(cam, cmap="jet", alpha=0.5)
    axes[0, k].axis("off"); axes[1, k].axis("off")
plt.tight_layout(); plt.show()


---
## Global hints
- Get the whole pipeline running on a **small subset** before scaling up — a model that trains badly beats a pipeline that won't run.
- "Better" only means something relative to your Part C baseline.
- Don't let test data leak into training or into augmentation statistics.
- `CrossEntropyLoss` wants raw logits; don't add a softmax before it.
- PyTorch: `model.train()` vs `model.eval()`, and `optimizer.zero_grad()` every step.


## What to submit & grading
**Submit** this completed notebook (code + outputs visible) and a short report (≈1–2 pages): preprocessing choices, the Part D comparison table, your confusion-matrix analysis, and a reflection on what helped most. State your **final test accuracy**.

| Component | Weight |
|---|---|
| A — Data exploration | 10% |
| B — Preprocessing | 10% |
| C — Baseline CNN | 25% |
| D — Improvements + comparison table | 30% |
| E — Evaluation & error analysis | 15% |
| Report clarity & reflection | 10% |
| F — Stretch goals | +10% bonus |

Grades reward **deliberate, measured** improvement and clear reasoning — not just the highest accuracy. A modest CNN reaches ~95%+ test accuracy here; well-tuned models exceed 98–99%.
